<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/VIX_SPIKE_SCAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX Spike Scan v1 — Features ciblées « fortes hausses » sur l'univers lourd

**Contexte.** La validation walk-forward (`VIX_CHAMPION_WF`) a établi que le **RandomForest GLOBAL h=5j** est le modèle unique robuste (F1_dir≈0.61 ±0.016), mais qu'il **rate les fortes hausses du VIX** (F1_UP_FORT≈0.36). Le champion STRESS du rapport, lui, s'effondre en walk-forward → requalifié en artefact de split statique.

**Objectif.** Tester si de nouvelles **features ciblées spike** (rough volatility / Hurst, semi-variance haussière, structure par terme VIX, indice SKEW, prime de vol-of-vol, sauts signés) améliorent la détection des fortes hausses **sans dégrader** la direction.

**Protocole.**
- **Univers lourd** : 339 tickers (repris du notebook « Institutional framework ») + tickers spike (^SKEW, ^VIX3M, ^VIX9D) + FRED curé.
- Features de base (Kalman/HMM/EGARCH/Heston/VRP/Hawkes, versions causales v2) **+ features spike** (toutes causales, cf. cellule dédiée).
- **Scan walk-forward complet** (5 folds) sur la grille : régime ∈ {GLOBAL, CALM, NORMAL, STRESS} × N ∈ {5,…,15} features × algo ∈ {XGBoost, LightGBM, RandomForest}.
- Sélection SHAP **refaite dans chaque fold** (sur son train uniquement) — anti-fuite (R1/R2).
- On observe les meilleurs résultats et on mesure **combien de features spike** les meilleurs modèles retiennent (preuve directe de leur utilité).

**Verdict attendu.** Si les meilleurs modèles WF sélectionnent des features spike et que F1_UP_FORT monte au-dessus de 0.36 (idéalement les 3 métriques > 0.50 simultanément, robustement), les features spike sont retenues. Sinon, elles sont écartées.


In [ ]:
import subprocess, sys
pkgs = ['xgboost','lightgbm','yfinance','pandas_datareader','arch',
        'pykalman','hmmlearn','shap','xlsxwriter','imbalanced-learn',
        'statsmodels','pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


In [ ]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import shap
from scipy.stats import multivariate_normal
from scipy.special import logsumexp
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib
import statsmodels.api as sm

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_SPIKE_SCAN'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'start_date': '2000-01-01',
    'flat_thr': 0.003,
    'horizon': 5,                 # h=5j : horizon le plus stable en walk-forward (rapport)
    'n_wf_folds': 5,
    'n_features_grid': list(range(5, 16)),   # scan N de 5 à 15 features
    'regimes': ['GLOBAL', 'CALM', 'NORMAL', 'STRESS'],
    'algos': ['XGBoost', 'LightGBM', 'RandomForest'],   # top-3 performeurs
    'min_train_frac': 0.40,       # 1er fold : 40% de l'historique en train
    'yf_coverage': 0.85,          # filtre couverture univers lourd
    'spike_coverage': 0.30,       # tickers spike récents (VIX3M/VIX9D depuis 2007/2011)
    'shap_sample': 500,           # échantillon max pour l'explainer SHAP
    'pool_prefilter': 400,        # pré-filtre du pool par gain XGBoost avant SHAP (perf)
}
TARGET_COL = 'VIX_Amplitude_Class'

# Univers lourd (339 tickers) — repris du notebook "Institutional framework"
YF_TICKERS = """
^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX ^FTSE ^N225
^HSI ^GDAXI ^FCHI ^STOXX50E SPY QQQ TLT GLD USO UUP FXE FXY
HYG LQD AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA
PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT
VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY
DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA
^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF
XLE XLV XLU XLP XLI XLY XLRE XLB XLC GOOGL META AVGO
ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT
EQIX CCI PSA ABBV MRK BMY AMGN GILD BNTX MRNA CRSP VRTX
ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB
EOG MPC PSX VLO PM MO BTI BP TTE ENB MET ADM
MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL YUM
QSR DPZ BLMN NWL RRR DASH LYFT UBER TGT M LOW ROST
BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR
VOD TM LOGI NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC
CORN SOYB IEF SHY SHV BIL AGG BND JNK VCIT VCSH EMB
MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP
EWI EWQ EWT EWY EWZ EWC EWS EWM FXI MCHI IEMG EEM
VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA
FXC FXF CEW CYB BZF FXD FXN GBTC COIN MSTR BITO MARA
RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV VTV VUG VB
SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD
ARKK XBI SOXX IBB IYT XHB KRE KBE ITA XOP OIH IYM
PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII LDOS EBAY
MELI SHOP SE PDD JD VIPS UPST RBLX SNOW CRWD ZM ROKU
PINS SNAP SPCE
""".split()

# Tickers ciblés "spike / tail-risk" ajoutés pour cette expérience :
#   ^SKEW  = CBOE SKEW index (prix du risque de queue via options SPX OTM)
#   ^VIX3M = VIX 3 mois (structure par terme : backwardation = stress)
#   ^VIX9D = VIX 9 jours (front de la structure par terme)
SPIKE_TICKERS = "^SKEW ^VIX3M ^VIX9D".split()

# FRED : jeu curé validé par SHAP dans le rapport (NFCI = feature la plus stable)
FRED_SERIES = {'NFCI': 'NFCI', 'STLFSI': 'STLFSI4', 'T10Y2Y': 'T10Y2Y', 'EFFR': 'EFFR'}

# Références du rapport (batch EGARCH, split statique 80/20) — SANS features spike,
# à confronter aux meilleurs résultats walk-forward AVEC features spike.
REPORT_REFS = {
    'GLOBAL h=5j (batch, statique)':  {'F1_dir': 0.6232},
    'GLOBAL h=5j RF (walk-forward)':  {'F1_dir': 0.6085, 'F1_dir_std': 0.0218, 'F1_UP_FORT': 0.360},
    'STRESS h=5j (batch, statique)':  {'F1_dir': 0.5871, 'F1_UP_FORT': 0.3768, 'F1_DOWN_FORT': 0.5182},
}

print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | univers={len(YF_TICKERS)} tickers "
      f"+ {len(SPIKE_TICKERS)} spike | régimes={CONFIG['regimes']} | N={CONFIG['n_features_grid'][0]}..{CONFIG['n_features_grid'][-1]}")


## Fondements — Validation walk-forward et sélection de features sans fuite

### Walk-forward (expanding window)
**Principe.** Au lieu d'un unique split train/test, on découpe l'axe temporel en $K$ folds successifs : à chaque fold $k$, on entraîne sur *tout* l'historique disponible jusqu'à une date $t_k$ (fenêtre croissante) et on teste sur la période suivante $]t_k, t_{k+1}]$. On obtient $K$ mesures de performance hors-échantillon au lieu d'une seule.

**Pourquoi c'est plus fiable qu'un split statique.** Un split unique 80/20 mesure la performance sur *une seule* période de test (ici ~2021-2026) : le score dépend fortement des régimes de marché qu'elle contient. Le walk-forward donne une **distribution** de scores ($\bar{F_1} \pm \hat\sigma$) sur des périodes différentes — la moyenne estime la performance attendue, l'écart-type mesure la **stabilité**. Un modèle à 0.62 statique mais 0.55±0.10 en WF est moins bon qu'un modèle à 0.60 statique et 0.60±0.02 en WF.

**Ce qu'on ne fait jamais** (règles absolues du rapport) : shuffle (l'ordre temporel est l'information), fit d'un estimateur (scaler, quantiles de cible, EGARCH, HMM...) sur des données postérieures au train du fold.

### Sélection de features par fold — le piège de la fuite de sélection
Si l'on sélectionne les $N$ meilleures features par SHAP **une fois sur tout l'historique**, puis qu'on "valide" en walk-forward, la sélection a déjà vu les périodes de test : les features retenues sont précisément celles qui marchaient sur ces périodes. Ce biais (*selection leakage* / double usage des données) suffit à gonfler artificiellement les scores WF.

**Protocole correct (appliqué ici).** À chaque fold : (1) un modèle pilote XGBoost est entraîné sur le train du fold uniquement ; (2) les valeurs de SHAP (contribution marginale moyenne de chaque feature à la prédiction, fondée sur la valeur de Shapley de la théorie des jeux coopératifs : $\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F|-|S|-1)!}{|F|!}[f(S \cup \{i\}) - f(S)]$) classent les features ; (3) les $N$ premières sont retenues pour ce fold. La liste peut donc **changer d'un fold à l'autre** — c'est voulu, et la stabilité de cette liste entre folds est elle-même un diagnostic (des features qui changent complètement à chaque fold = signal fragile).

### Métriques hiérarchiques (rappel)
- **Niveau 1** $F_1^{dir}$ : moyenne des F1 binaires UP/DOWN après agrégation des 4 classes en 2 directions.
- **Niveau 2** $F_1^{UP\_FORT}$ : F1 de FORT vs FAIBLE, calculé *uniquement* sur le sous-ensemble des vrais UP.
- **Niveau 3** $F_1^{DOWN\_FORT}$ : idem sur le sous-ensemble des vrais DOWN.

Objectif du champion : les trois > 0.50 simultanément, en walk-forward cette fois.


In [ ]:
def load_data(start=CONFIG['start_date']):
    t0 = time.time()
    # --- univers lourd (filtre de couverture standard) ---
    raw = yf.download(YF_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
    raw.columns = [c.replace('^', 'IDX_').replace('-', '_') for c in raw.columns]
    raw = raw.loc[:, raw.notna().mean() >= CONFIG['yf_coverage']].ffill().dropna(how='all')
    print(f"  YF lourd: {raw.shape[0]}j × {raw.shape[1]} retenus (couv≥{CONFIG['yf_coverage']}) ({time.time()-t0:.1f}s)")

    # --- tickers spike/tail-risk (couverture plus basse : séries récentes) ---
    try:
        sp = yf.download(SPIKE_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
        if isinstance(sp, pd.Series): sp = sp.to_frame()
        sp.columns = [c.replace('^', 'IDX_').replace('-', '_') for c in sp.columns]
        sp = sp.loc[:, sp.notna().mean() >= CONFIG['spike_coverage']]
        keep = [c for c in sp.columns if c not in raw.columns]
        if keep:
            raw = pd.concat([raw, sp[keep].reindex(raw.index).ffill()], axis=1)
            print(f"  YF spike: {keep} ajoutés")
        else:
            print("  YF spike: aucun ticker spike disponible (couverture insuffisante)")
    except Exception as e:
        print(f"  [WARN] spike tickers: {e}")

    # --- FRED ---
    fred_list = []
    for name, sid in FRED_SERIES.items():
        try:
            s = web.DataReader(sid, 'fred', start).squeeze(); s.name = f'FRED_{name}'
            fred_list.append(s)
        except Exception as e: print(f"  [WARN] {sid}: {e}")
    if fred_list:
        raw = pd.concat([raw, pd.concat(fred_list, axis=1).reindex(raw.index, method='ffill')], axis=1)
    print(f"  Total: {raw.shape} ({time.time()-t0:.1f}s)")
    return raw

df_raw = load_data()
all_dates = df_raw.dropna(how='all').index.sort_values()
VIX_COL = [c for c in df_raw.columns if ('IDX_VIX' in c or c.endswith('_VIX')) and 'VXN' not in c
           and 'VVIX' not in c and 'VIX3M' not in c and 'VIX9D' not in c][0]
SPX_COLS = [c for c in df_raw.columns if 'GSPC' in c or c == 'SPY']
SPX_COL = SPX_COLS[0] if SPX_COLS else None

# Bornes des folds walk-forward (expanding window)
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
FIT_IDX = FOLD_CUTS[0]

print(f"VIX={VIX_COL} | SPX={SPX_COL}")
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


## Fondements théoriques des features (Kalman, HMM, EGARCH/GJR, Heston, VRP, Hawkes)

La cellule suivante (`build_features`) construit plusieurs familles de features issues de modèles de séries temporelles financières. Pour chacune : le principe, l'objectif dans ce pipeline, et les fondements mathématiques.

### Filtre de Kalman
**Principe** — Estimateur récursif à variance minimale de l'état caché d'un système linéaire-gaussien à partir d'observations bruitées : état $x_t = F x_{t-1} + w_t$, observation $y_t = H x_t + v_t$, avec $w_t \sim \mathcal N(0,Q)$, $v_t \sim \mathcal N(0,R)$. Deux étapes à chaque pas : **prédiction** ($\hat x_{t|t-1} = F\hat x_{t-1}$) puis **mise à jour** par le gain de Kalman $K_t = P_{t|t-1}H^\top(HP_{t|t-1}H^\top+R)^{-1}$, qui pondère l'innovation $y_t - H\hat x_{t|t-1}$.
**Objectif ici** — Le VIX est modélisé comme une marche aléatoire bruitée ($F=H=1$) : le filtre en extrait un niveau "sous-jacent" débruité de façon **causale** (chaque estimation ne dépend que du passé). Le résidu (VIX observé − VIX filtré) sert de feature d'écart au régime.
**Point de vigilance** — Le lisseur RTS (`kf.smooth()`, forward-backward) utilise aussi les observations *futures* : c'est non causal, donc écarté en walk-forward (cf. fix v2).

### HMM (Hidden Markov Model) — détection de régime
**Principe** — Chaîne de Markov cachée à états discrets (ici 2 : "calme"/"stress") qui gouverne la loi des observations à chaque instant. Paramètres : probabilités initiales $\pi$, matrice de transition $A$, lois d'émission par état (ici gaussiennes). Apprentissage par EM (Baum-Welch) : l'étape E calcule les probabilités a posteriori des états via l'algorithme forward-backward, l'étape M réestime $\pi, A$ et les émissions.
**Objectif ici** — Estimer $P(\text{état stress}_t \mid \text{observations})$ comme feature de régime de volatilité.
**Fondements / cours de référence** — [Probabilistic Graphical Models (Master MVA)](https://www.master-mva.com/cours/probabilistic-graphical-models/) : distinction clé entre **filtrage** (forward seul, $P(z_t\mid x_{1:t})$, causal) et **lissage** (forward-backward, $P(z_t\mid x_{1:T})$, non causal — utilise le futur). `hmmlearn.predict_proba()` fait du lissage : c'est la fuite corrigée en v2 via `hmm_filtered_proba()` (forward uniquement).

### EGARCH / GJR-GARCH — volatilité conditionnelle
**Principe** — Modélisent la variance conditionnelle $\sigma_t^2$ d'une série de rendements pour capter le *clustering* de volatilité et l'**effet de levier** (une baisse augmente plus la volatilité future qu'une hausse de même ampleur). GARCH(1,1) : $\sigma_t^2=\omega+\alpha r_{t-1}^2+\beta\sigma_{t-1}^2$. **EGARCH** (Nelson, 1991) modélise $\log\sigma_t^2$ (positivité garantie sans contrainte sur les paramètres) avec un terme asymétrique en $r_{t-1}/\sigma_{t-1}$. **GJR-GARCH** (Glosten–Jagannathan–Runkle, 1993) ajoute un terme indicatrice sur les rendements négatifs.
**Objectif ici** — Paramètres estimés sur les rendements du S&P 500 en train uniquement, puis le chemin de variance conditionnelle est recalculé de façon causale (récursion sur les rendements passés) sur tout l'historique via `arch_model.fix()` — fixe le bug v1 où les features de test restaient plates.

### Modèle de Heston (proxy)
**Principe** — Modèle à volatilité stochastique : la variance instantanée suit un processus CIR (Cox–Ingersoll–Ross) à retour à la moyenne $dv_t=\kappa(\theta-v_t)dt+\xi\sqrt{v_t}\,dW_t$, corrélé au prix ($\rho$). $\kappa$ = vitesse de retour à la moyenne, $\theta$ = variance long-terme, $\xi$ = vol-of-vol.
**Objectif ici** — Sans données d'options pour calibrer Heston par ajustement de surface de volatilité implicite, on construit des **proxies** : $\theta$ via variance réalisée glissante, $\xi$ via le VVIX (vol du VIX), $\kappa$ via la demi-vie d'un AR(1) glissant sur le VIX (relation $HL=\ln 2/\kappa$). L'espérance conditionnelle de variance à horizon $h$, $\mathbb E[v_{t+h}]=\theta+(v_t-\theta)e^{-\kappa h}$, sert de feature de "retour à la moyenne attendu".

### VRP (Variance Risk Premium)
**Principe** — Écart entre variance implicite (ici $(\text{VIX}/100)^2$) et variance réalisée future attendue : $VRP_t = IV_t^2-\mathbb E[RV_{t\to t+h}]$ — mesure la prime que le marché paie pour se couvrir contre le risque de variance.
**Objectif ici** — $\mathbb E[RV]$ est estimée par régression OLS de la variance réalisée future sur des variances réalisées passées à plusieurs échelles (1j/5j/22j), une version simplifiée de l'approche **HAR** (Heterogeneous AutoRegressive, Corsi 2009).

### Processus de Hawkes (proxy d'intensité de sauts)
**Principe** — Processus ponctuel auto-excitant : chaque événement (ici un "saut" de rendement du VIX) augmente temporairement l'intensité d'occurrence de futurs événements, avec décroissance exponentielle $\lambda(t)=\mu+\sum_{t_i<t}\alpha e^{-\beta(t-t_i)}$ — modélise le *clustering* temporel des chocs de volatilité.
**Objectif ici** — Proxy simplifié (pas de MLE complet du processus ponctuel) : intensité pondérée par décroissance exponentielle du temps écoulé depuis les sauts passés.


In [ ]:
def hmm_filtered_proba(model, X):
    """Probabilités d'état HMM causales (forward filtering P(state_t | obs_1..t)),
    contrairement à predict_proba() qui fait du forward-backward (smoothing) et
    utilise donc des observations futures -> fuite de données en walk-forward."""
    n = len(X)
    n_states = model.n_components
    logframe = np.zeros((n, n_states))
    for s in range(n_states):
        cov = model.covars_[s]
        if cov.ndim == 2:
            cov = cov + np.eye(cov.shape[0]) * 1e-6
        logframe[:, s] = multivariate_normal.logpdf(X, mean=model.means_[s], cov=cov)
    log_start = np.log(model.startprob_ + 1e-300)
    log_trans = np.log(model.transmat_ + 1e-300)
    fwd = np.zeros((n, n_states))
    fwd[0] = log_start + logframe[0]
    fwd[0] -= logsumexp(fwd[0])
    for t in range(1, n):
        for j in range(n_states):
            fwd[t, j] = logsumexp(fwd[t - 1] + log_trans[:, j]) + logframe[t, j]
        fwd[t] -= logsumexp(fwd[t])
    return np.exp(fwd)


def build_features(df_raw, vix_col, spx_col, split_idx):
    t0 = time.time(); feats = {}
    vix = df_raw[vix_col].replace([np.inf, -np.inf], np.nan).ffill().bfill()
    vix_ret = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)
    spx_ret = pd.Series(0., index=df_raw.index)
    if spx_col:
        spx = df_raw[spx_col].ffill().bfill()
        spx_ret = np.log(spx / spx.shift(1)).fillna(0)

    # Rendements multi-horizons pour tous les tickers
    print(f"  [FEAT] Rendements ({df_raw.shape[1]} séries)...")
    for col in df_raw.columns:
        s = df_raw[col].replace([np.inf, -np.inf], np.nan).ffill().bfill()
        lr = np.log(s / s.shift(1)).replace([np.inf, -np.inf], np.nan)
        for w in [1, 5, 20]:
            feats[f'{col}_ret_{w}d'] = np.log(s / s.shift(w)).replace([np.inf, -np.inf], np.nan)
        feats[f'{col}_vol_20d'] = lr.rolling(20, min_periods=10).std()
        mu = s.rolling(60, min_periods=30).mean(); sd = s.rolling(60, min_periods=30).std().replace(0, np.nan)
        feats[f'{col}_zscore_60d'] = (s - mu) / sd

    # VIX features
    for w in [5, 10, 20]:
        ma = vix.rolling(w, min_periods=w // 2).mean(); sd = vix.rolling(w, min_periods=w // 2).std().replace(0, np.nan)
        feats[f'vix_zscore_{w}d'] = (vix - ma) / sd
        feats[f'vix_vs_ma{w}'] = (vix - ma) / ma.replace(0, np.nan)
    feats['vix_level'] = vix; feats['vix_ma_20'] = vix.rolling(20, min_periods=10).mean()
    for w in [5, 10]: feats[f'vix_vol_of_vol_{w}d'] = vix_ret.rolling(w, min_periods=w // 2).std()
    for w in [2, 3, 5]: feats[f'vix_momentum_{w}d'] = vix.pct_change(w)
    feats['vix_acceleration_1d'] = vix_ret - vix_ret.shift(1)
    feats['vix_acceleration_3d'] = vix_ret - vix_ret.shift(3)
    feats['vix_erratic_ratio'] = vix_ret.abs().rolling(5, min_periods=3).max() / vix_ret.abs().rolling(5, min_periods=3).mean().replace(0, np.nan)
    feats['vix_vol_ratio_5_60'] = vix_ret.rolling(5, min_periods=3).std() / vix_ret.rolling(60, min_periods=30).std().replace(0, np.nan)
    feats['vix_max_abs_ret_5d'] = vix_ret.abs().rolling(5, min_periods=3).max()
    feats['vix_mean_abs_ret_5d'] = vix_ret.abs().rolling(5, min_periods=3).mean()
    if spx_col:
        spx = df_raw[spx_col].ffill().bfill()
        feats['spx_drawdown_252d'] = (spx - spx.rolling(252, min_periods=126).max()) / spx.rolling(252, min_periods=126).max().replace(0, np.nan)
        feats['spx_vol_5d'] = spx_ret.rolling(5, min_periods=3).std()
        feats['spx_abs_ret_max_5d'] = spx_ret.abs().rolling(5, min_periods=3).max()
        feats['spx_momentum_3d'] = spx.pct_change(3)
        feats['vix_spx_corr_30d'] = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    # EGARCH + GJR-GARCH
    # [FIX v2] Params estimés UNIQUEMENT sur le train (pas de fuite), puis appliqués
    # via .fix() sur toute la série pour obtenir un chemin de variance conditionnelle
    # causal (récursif, ne dépend que du passé) sur train+test. v1 utilisait
    # res.forecast(start=0) qui ne s'étend pas au-delà de l'échantillon d'estimation
    # -> les features EGARCH du test étaient plates (ffill de la dernière valeur train).
    print(f"  [FEAT] EGARCH ({time.time() - t0:.0f}s)...")
    try:
        sp_tr = (spx_ret.iloc[:split_idx] * 100)
        sp_full = (spx_ret * 100).replace([np.inf, -np.inf], np.nan).fillna(0)
        am = arch_model(sp_tr, vol='EGARCH', p=1, q=1, dist='skewt', rescale=False)
        res = am.fit(disp='off', show_warning=False)
        am_full = arch_model(sp_full, vol='EGARCH', p=1, q=1, dist='skewt', rescale=False)
        res_full = am_full.fix(res.params)
        cv = (res_full.conditional_volatility.pow(2) / 10000).reindex(df_raw.index).replace([np.inf, -np.inf], np.nan)
        feats['egarch_condvar'] = cv; feats['egarch_delta'] = cv.diff()
        for h in [1, 3, 5]:
            feats[f'EGARCH_SPX_condvar_h{h}'] = cv; feats[f'EGARCH_SPX_delta_h{h}'] = cv.diff()
        # GJR-GARCH
        am2 = arch_model(sp_tr, vol='GARCH', p=1, o=1, q=1, dist='skewt', rescale=False)
        res2 = am2.fit(disp='off', show_warning=False)
        am2_full = arch_model(sp_full, vol='GARCH', p=1, o=1, q=1, dist='skewt', rescale=False)
        res2_full = am2_full.fix(res2.params)
        cv2 = (res2_full.conditional_volatility.pow(2) / 10000).reindex(df_raw.index).replace([np.inf, -np.inf], np.nan)
        feats['gjr_condvar'] = cv2; feats['gjr_delta'] = cv2.diff()
        print(f"    EGARCH+GJR OK (paramètres train-only, chemin causal complet)")
    except Exception as e: print(f"    [WARN] GARCH: {e}")

    # Kalman (causal ; +1j shift anti-leakage)
    # [FIX v2] v1 calculait aussi kf.smooth() (RTS smoother, non causal : l'estimé à la
    # date t dépend d'observations futures jusqu'à la fin de l'échantillon) et l'utilisait
    # pour VIX_Innovation -> fuite. On ne garde que le filtre forward (causal).
    print(f"  [FEAT] Kalman ({time.time() - t0:.0f}s)...")
    try:
        vc = vix.interpolate('linear').ffill().bfill().astype(float)
        kf = KalmanFilter(transition_matrices=[[1.]], observation_matrices=[[1.]],
                        initial_state_mean=[float(vc.iloc[0])],
                        initial_state_covariance=[[1.]],
                        em_vars=['transition_covariance', 'observation_covariance'])
        kf = kf.em(vc.iloc[:split_idx].values.reshape(-1, 1), n_iter=20)
        smf, _ = kf.filter(vc.values.reshape(-1, 1))
        kf_f = pd.Series(smf[:, 0], index=df_raw.index)
        feats['VIX_Residual'] = (vc - kf_f).shift(1).replace([np.inf, -np.inf], np.nan)
        feats['VIX_Innovation'] = (vc - kf_f.shift(1)).replace([np.inf, -np.inf], np.nan)
        feats['kalman_residual'] = feats['VIX_Residual']
        feats['kalman_innovation'] = feats['VIX_Innovation']
        feats['kalman_filtered'] = kf_f
        print(f"    Kalman OK (Q={kf.transition_covariance[0,0]:.5f}, filtre causal uniquement)")
    except Exception as e: print(f"    [WARN] Kalman: {e}")

    # HMM K=2 (probabilités filtrées causales, pas smoothées)
    # [FIX v2] mh.predict_proba()/mh.predict() en v1 utilisaient l'algorithme forward-
    # backward sur train+test entier -> l'état "stress" à la date t incorporait de
    # l'information sur les rendements futurs (t+1..T). On calcule maintenant les
    # probabilités filtrées (forward only) via hmm_filtered_proba(), strictement causales.
    print(f"  [FEAT] HMM ({time.time() - t0:.0f}s)...")
    try:
        rv5 = vix_ret.pow(2).rolling(5, min_periods=3).mean()
        mu_t = vix.iloc[:split_idx].mean(); sd_t = vix.iloc[:split_idx].std()
        vix_n = (vix - mu_t) / (sd_t if sd_t > 1e-8 else 1)
        X_hmm = pd.DataFrame({'r': vix_ret, 'v': np.sqrt(rv5.clip(0)), 'l': vix_n}).dropna()
        mh = hmmlib.GaussianHMM(n_components=2, covariance_type='full', n_iter=200, random_state=SEED)
        mh.fit(X_hmm.iloc[:split_idx].values)
        st = mh.predict(X_hmm.iloc[:split_idx].values)
        rv_v = [rv5.reindex(X_hmm.index[:split_idx]).values[st == s].mean() if (st == s).any() else 0 for s in range(2)]
        ss = int(np.argmax(rv_v))
        pr = hmm_filtered_proba(mh, X_hmm.values)
        p_stress = pd.Series(pr[:, ss], index=X_hmm.index).reindex(df_raw.index)
        feats['P_stress_HMM'] = p_stress; feats['hmm_p_stress'] = p_stress
        feats['hmm_state'] = pd.Series(pr.argmax(axis=1), index=X_hmm.index).reindex(df_raw.index)
        print(f"    HMM OK (état stress={ss}, probabilités filtrées causales)")
    except Exception as e: print(f"    [WARN] HMM: {e}")

    # Heston proxies + espérances conditionnelles
    print(f"  [FEAT] Heston ({time.time() - t0:.0f}s)...")
    try:
        v0 = (vix / 100).pow(2); theta = vix_ret.pow(2).rolling(60, min_periods=30).mean()
        vvix_c = [c for c in df_raw.columns if 'VVIX' in c]
        xi = (df_raw[vvix_c[0]].ffill().bfill() / 100) if vvix_c else vix_ret.rolling(20).std()
        xi = xi.reindex(df_raw.index, method='ffill').replace([np.inf, -np.inf], np.nan)
        rho = vix_ret.rolling(30, min_periods=15).corr(spx_ret)
        rho_60 = vix_ret.rolling(60, min_periods=30).corr(spx_ret)
        # Kappa via half-life AR(1) rolling 252j
        def rolling_kappa(s, w=252):
            k = pd.Series(np.nan, index=s.index); sf = s.ffill().bfill()
            for i in range(w, len(sf)):
                try:
                    b = np.corrcoef(sf.iloc[i - w:i].values, sf.iloc[i - w + 1:i + 1].values)[0, 1]
                    if np.isfinite(b) and 0 < abs(b) < 0.9999:
                        hl = -np.log(2) / np.log(abs(b))
                        if np.isfinite(hl) and hl > 0: k.iloc[i] = np.log(2) / hl
                except: pass
            return k.replace([np.inf, -np.inf], np.nan)
        kappa = rolling_kappa(vix)
        feats['heston_v0'] = v0; feats['heston_theta'] = theta
        feats['heston_xi'] = xi; feats['heston_rho'] = rho; feats['heston_rho_60'] = rho_60
        feats['heston_kappa'] = kappa
        feats['heston_feller'] = (2 * kappa * theta) / xi.pow(2).replace(0, np.nan)
        feats['heston_v0_minus_theta'] = v0 - theta
        mu_xi = xi.iloc[:split_idx].mean(); sd_xi = xi.iloc[:split_idx].std()
        feats['heston_xi_zscore'] = (xi - mu_xi) / (sd_xi if sd_xi > 1e-8 else 1)
        feats['heston_xi_ma5'] = xi.rolling(5, min_periods=3).mean()
        mu_k = kappa.iloc[:split_idx].mean(); sd_k = kappa.iloc[:split_idx].std()
        feats['heston_kappa_zscore'] = (kappa - mu_k) / (sd_k if sd_k > 1e-8 else 1)
        for h in [1, 3, 5, 7, 10]:
            ev = (theta + (v0 - theta) * np.exp(-kappa * h)).replace([np.inf, -np.inf], np.nan)
            var_ev = (v0 * xi ** 2 * np.exp(-kappa * h) * (1 - np.exp(-kappa * h)) / kappa.replace(0, np.nan)
                    + theta * xi ** 2 * (1 - np.exp(-kappa * h)) ** 2 / (2 * kappa.replace(0, np.nan))).replace([np.inf, -np.inf], np.nan)
            feats[f'heston_ev_h{h}'] = ev
            feats[f'heston_spread_h{h}'] = v0 - ev
            feats[f'heston_vol_h{h}'] = np.sqrt(ev.clip(lower=0)) * 100
            feats[f'heston_var_ev_h{h}'] = var_ev
        print(f"    Heston OK (xi_mean={xi.iloc[:split_idx].mean():.4f})")
    except Exception as e: print(f"    [WARN] Heston: {e}")

    # VRP
    # [FIX v2] La cible d'entraînement de la régression (rv_tgt) regarde 22j dans le
    # futur. En v1, htr=hdf.iloc[:split_idx] incluait des lignes dont la cible dépassait
    # split_idx et lisait donc des données de test -> fuite dans les coefficients OLS.
    # On tronque désormais l'échantillon d'entraînement de 22j avant le split.
    print(f"  [FEAT] VRP ({time.time() - t0:.0f}s)...")
    try:
        rv1 = vix_ret.pow(2).replace([np.inf, -np.inf], np.nan)
        rv5d = rv1.rolling(5, min_periods=3).mean(); rv22d = rv1.rolling(22, min_periods=10).mean()
        rv_tgt = rv1.shift(-22).rolling(22, min_periods=11).mean()
        hdf = pd.DataFrame({'rv1': rv1, 'rv5': rv5d, 'rv22': rv22d, 'y': rv_tgt}).dropna().replace([np.inf, -np.inf], np.nan).dropna()
        train_cutoff = max(split_idx - 22, 1)
        htr = hdf.iloc[:train_cutoff]
        Xh = sm.add_constant(htr[['rv1', 'rv5', 'rv22']], has_constant='add')
        hm = sm.OLS(htr['y'], Xh).fit()
        Xf = sm.add_constant(hdf[['rv1', 'rv5', 'rv22']], has_constant='add').fillna(0)
        rv_pred = hm.predict(Xf).reindex(df_raw.index).fillna(0)
        vrp = (vix / 100).pow(2) - rv_pred
        mu_v = vrp.iloc[:split_idx].mean(); sd_v = vrp.iloc[:split_idx].std()
        feats['VRP'] = vrp; feats['VRP_zscore'] = (vrp - mu_v) / (sd_v if sd_v > 1e-8 else 1)
        feats['VRP_ma5'] = vrp.rolling(5, min_periods=3).mean()
        print(f"    VRP OK (R²={hm.rsquared:.4f}, cible tronquée avant le split)")
    except Exception as e: print(f"    [WARN] VRP: {e}")

    # Jump + Hawkes
    print(f"  [FEAT] Jump+Hawkes ({time.time() - t0:.0f}s)...")
    sig60 = vix_ret.rolling(60, min_periods=30).std()
    is_j = (vix_ret.abs() > 3 * sig60).astype(float)
    feats['jump_intensity_20d'] = is_j.rolling(20, min_periods=10).mean()
    feats['jump_intensity_60d'] = is_j.rolling(60, min_periods=30).mean()
    try:
        sig_hw = vix_ret.rolling(30, min_periods=15).std()
        jt = vix_ret.index[vix_ret.abs() > 2 * sig_hw]
        hw = pd.Series(0., index=vix_ret.index)
        for i, t in enumerate(vix_ret.index):
            past = jt[jt < t]
            hw.iloc[i] = 0.3 + 0.3 * float(np.sum(np.exp(-0.1 * np.array([(t - tj).days for tj in past], dtype=float)))) if len(past) else 0.3
        mu_hw = hw.iloc[:split_idx].mean(); sd_hw = hw.iloc[:split_idx].std()
        feats['hawkes_intensity'] = hw; feats['hawkes_zscore'] = (hw - mu_hw) / (sd_hw if sd_hw > 1e-8 else 1)
    except Exception as e: print(f"    [WARN] Hawkes: {e}")

    # Implied correlation proxy
    try:
        sec = [c for c in df_raw.columns if any(s in c for s in ['XLK', 'XLF', 'XLE', 'XLV', 'XLU', 'XLB', 'XLI', 'XLY'])]
        if sec:
            vix_sq = (vix / 100).pow(2); w = 1. / len(sec)
            sv_sum = sum(w ** 2 * np.log(df_raw[c].ffill() / df_raw[c].ffill().shift(1)).rolling(21, min_periods=10).std().pow(2) for c in sec)
            feats['impl_corr_proxy'] = (vix_sq - sv_sum).clip(-1, 1)
    except: pass

    df_feat = pd.DataFrame(feats, index=df_raw.index).replace([np.inf, -np.inf], np.nan)
    df_full = pd.concat([df_raw, df_feat], axis=1)
    df_full = df_full.loc[:, ~df_full.columns.duplicated()]
    print(f"  [FEAT] Total: {df_full.shape[1]} cols ({time.time() - t0:.0f}s)")
    return df_full


# [WF] Les estimateurs internes (EGARCH, Kalman EM, HMM, OLS du VRP, moyennes/écarts
# de normalisation) sont fittés sur le train du PREMIER fold uniquement (FIT_IDX) :
# comme tous les folds walk-forward ont un train qui commence au début de
# l'historique, aucune de ces estimations ne voit jamais une période de test.
print("[FEATURES] Démarrage (fit des estimateurs sur le train du 1er fold)...")
df_features = build_features(df_raw, VIX_COL, SPX_COL, FIT_IDX)
print(f"Dataset: {df_features.shape}")


## Nouvelles features ciblées « spike » (fortes hausses du VIX)

Le modèle GLOBAL h=5j validé en walk-forward est robuste sur la direction (F1_dir≈0.61) mais **rate les fortes hausses** (F1_UP_FORT≈0.36). Ces features visent spécifiquement les précurseurs de spike de volatilité. **Toutes sont causales par construction** (ratios auto-normalisés ou z-scores sur fenêtre glissante) : chaque valeur à la date *t* n'utilise que le passé, donc aucune fuite train/test — pas besoin de les fitter sur le train.

### Rough volatility — exposant de Hurst (Gatheral, Jaisson & Rosenbaum, 2018)
La volatilité est « rugueuse » : l'exposant de Hurst $H$ des log-variances est empiriquement $\approx 0.1 \ll 0.5$. Un $H$ faible = trajectoire très irrégulière, forte anti-persistance = régime propice aux sauts. Estimé en fenêtre glissante via la loi d'échelle de la variance agrégée : pour des blocs de taille $L$, $\operatorname{Var}(\sum_L r) \propto L^{2H}$, d'où $H = \tfrac12\,\text{pente}\big(\log \operatorname{Var}(\sum_L r)\ \text{vs}\ \log L\big)$.

### Semi-variance réalisée haussière (Barndorff-Nielsen et al., 2010)
Décomposition de la variance réalisée en parts haussière/baissière : $RS^+ = \sum_{t} r_t^2\,\mathbb{1}(r_t>0)$. Le ratio $RS^+/RV$ mesure l'**asymétrie de la variance vers le haut** — une variance dominée par les hausses du VIX précède les spikes.

### Structure par terme du VIX (contango / backwardation)
En régime calme, la courbe VIX est en *contango* (VIX9D < VIX < VIX3M). Un passage en *backwardation* (front > long, ratio > 1) est un signal de stress classique. Features : $\text{VIX9D}/\text{VIX}$ et $\text{VIX}/\text{VIX3M}$ (ratios, sans normalisation, donc causaux). Construites seulement si `^VIX9D` / `^VIX3M` sont disponibles.

### Indice SKEW du CBOE (proxy options-flow / risque de queue)
Le SKEW mesure le prix payé pour les options SPX très en dehors de la monnaie (protection contre les krachs) : SKEW élevé = le marché price une queue gauche épaisse = demande de couverture. Features : niveau, variation à 5j, z-score glissant 252j. Construites seulement si `^SKEW` disponible.

### Prime de vol-of-vol (VVIX/VIX) et sauts signés
- **VVIX/VIX** : la volatilité du VIX rapportée à son niveau ; une vol-of-vol élevée relativement au VIX annonce l'instabilité.
- **Intensité de sauts haussiers** : fraction de jours où $r_t > 2\hat\sigma_{60}$ sur 20j, et asymétrie sauts hauts − sauts bas — mesure directe de la fréquence récente des chocs à la hausse.


In [ ]:
# ============================================================
# FEATURES SPIKE (toutes causales — aucune statistique fittée sur le train)
# ============================================================
def _hurst_window(x):
    """Exposant de Hurst d'une fenêtre de log-rendements via la loi d'échelle
    de la variance agrégée : Var(somme de L rendements) ∝ L^(2H)."""
    x = x[~np.isnan(x)]
    if len(x) < 40:
        return np.nan
    lags = [1, 2, 4, 8]; v = []
    for L in lags:
        n = len(x) // L
        if n < 4:
            return np.nan
        agg = x[:n * L].reshape(n, L).sum(axis=1)
        v.append(np.var(agg))
    v = np.array(v)
    if np.any(v <= 0):
        return np.nan
    slope = np.polyfit(np.log(lags), np.log(v), 1)[0]
    return slope / 2.0

def add_spike_features(df, vix_col):
    """Ajoute les features spike à df (in place) et renvoie la liste des noms ajoutés."""
    t0 = time.time(); added = []
    vix = df[vix_col].replace([np.inf, -np.inf], np.nan).ffill().bfill()
    r = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)

    def put(name, series):
        df[name] = series.replace([np.inf, -np.inf], np.nan)
        added.append(name)

    # 1. Semi-variance haussière (ratio RS+/RV), causal
    for w in [20, 60]:
        up = (r.clip(lower=0) ** 2).rolling(w, min_periods=w // 2).sum()
        tot = (r ** 2).rolling(w, min_periods=w // 2).sum().replace(0, np.nan)
        put(f'spike_semivar_up_ratio_{w}d', up / tot)

    # 2. Sauts signés (causal) : seuil 2×écart-type glissant 60j
    sig60 = r.rolling(60, min_periods=30).std()
    up_j = (r > 2 * sig60).astype(float); dn_j = (r < -2 * sig60).astype(float)
    put('spike_up_jump_int_20d', up_j.rolling(20, min_periods=10).mean())
    put('spike_jump_asym_20d', (up_j - dn_j).rolling(20, min_periods=10).mean())

    # 3. Rough volatility — Hurst glissant (causal)
    put('spike_hurst_120d', pd.Series(r.values, index=r.index)
        .rolling(120, min_periods=80).apply(_hurst_window, raw=True))

    # 4. Structure par terme du VIX (backwardation) — si disponibles
    v9 = [c for c in df.columns if 'VIX9D' in c]
    v3 = [c for c in df.columns if 'VIX3M' in c]
    if v9:
        s9 = df[v9[0]].replace([np.inf, -np.inf], np.nan).ffill()
        put('spike_ts_9d_over_vix', s9 / vix.replace(0, np.nan))
    if v3:
        s3 = df[v3[0]].replace([np.inf, -np.inf], np.nan).ffill()
        put('spike_vix_over_3m', vix / s3.replace(0, np.nan))

    # 5. Indice SKEW (options-flow / risque de queue) — si disponible
    sk = [c for c in df.columns if 'SKEW' in c]
    if sk:
        s = df[sk[0]].replace([np.inf, -np.inf], np.nan).ffill()
        put('spike_skew_level', s)
        put('spike_skew_ret_5d', s.pct_change(5))
        mu = s.rolling(252, min_periods=120).mean(); sd = s.rolling(252, min_periods=120).std().replace(0, np.nan)
        put('spike_skew_z_252d', (s - mu) / sd)

    # 6. Prime de vol-of-vol VVIX/VIX — si VVIX disponible
    vv = [c for c in df.columns if 'VVIX' in c]
    if vv:
        vvix = df[vv[0]].replace([np.inf, -np.inf], np.nan).ffill()
        ratio = vvix / vix.replace(0, np.nan)
        put('spike_vvix_vix_ratio', ratio)
        mu = ratio.rolling(252, min_periods=120).mean(); sd = ratio.rolling(252, min_periods=120).std().replace(0, np.nan)
        put('spike_vvix_vix_z_252d', (ratio - mu) / sd)

    print(f"  [SPIKE] {len(added)} features ajoutées: {added} ({time.time()-t0:.0f}s)")
    return added

SPIKE_FEATURES = add_spike_features(df_features, VIX_COL)
print(f"Dataset enrichi: {df_features.shape}")


In [ ]:
def build_target(vix_series, horizon, split_idx):
    """Cible 4 classes à seuils conditionnels au régime, quantiles fittés
    uniquement sur les split_idx premières observations (train du fold)."""
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

# Pool de features candidat = colonnes engineered uniquement (les prix bruts,
# non stationnaires, sont exclus de la sélection SHAP)
FEATURE_POOL = [c for c in df_features.columns if c not in df_raw.columns]
print(f"Pool de features candidates: {len(FEATURE_POOL)}")


In [ ]:
# ============================================================
# SCAN WALK-FORWARD : régime × N(5-15) × algo, pool = features de base + spike
# Sélection SHAP refaite sur le train de CHAQUE fold (anti-fuite).
# ============================================================
def get_clf(algo):
    if algo == 'XGBoost':
        return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                             eval_metric='mlogloss', objective='multi:softprob',
                             random_state=SEED, n_jobs=-1, verbosity=0)
    if algo == 'LightGBM':
        return LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                              num_leaves=31, min_child_samples=10, subsample=0.8,
                              class_weight='balanced', random_state=SEED, verbose=-1, n_jobs=-1)
    if algo == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1)
    raise ValueError(algo)

def shap_rank(X_tr, y_tr, pool_names, top_n, prefilter):
    """Classe le pool par importance SHAP (fit sur le train du fold uniquement).
    Pré-filtre d'abord par gain XGBoost pour accélérer SHAP sur un gros pool."""
    nf = X_tr.shape[1]
    if nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1,
                           objective='multi:softprob', eval_metric='mlogloss',
                           random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X_tr, y_tr)
        keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1,
                          objective='multi:softprob', eval_metric='mlogloss',
                          random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    arr = np.asarray(arr).ravel()
    order_local = np.argsort(arr)[::-1][:top_n]
    global_idx = keep[order_local]
    return [pool_names[i] for i in global_idx]

scan_rows = []; t0 = time.time(); n_done = 0
n_total = CONFIG['n_wf_folds'] * len(CONFIG['regimes']) * len(CONFIG['n_features_grid']) * len(CONFIG['algos'])
N_MAX = CONFIG['n_features_grid'][-1]

for k in range(CONFIG['n_wf_folds']):
    cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
    cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
    target, reg_s, _ = build_target(df_features[VIX_COL], CONFIG['horizon'], cut)
    X_pool = df_features[FEATURE_POOL].reindex(target.index)
    reg_al = reg_s.reindex(target.index).fillna('NORMAL')

    for regime in CONFIG['regimes']:
        tr_mask = (target.index < cut_date).copy()
        te_mask = ((target.index >= cut_date) & (target.index <= nxt_date)).copy()
        if regime != 'GLOBAL':
            tr_mask &= (reg_al == regime).values
            te_mask &= (reg_al == regime).values
        y_tr = target.loc[tr_mask].values.astype(int)
        y_te = target.loc[te_mask].values.astype(int)
        if len(y_tr) < 100 or len(y_te) < 20:
            continue

        sc = RobustScaler()
        X_tr = sc.fit_transform(X_pool.loc[tr_mask].fillna(0).values)
        X_te = sc.transform(X_pool.loc[te_mask].fillna(0).values)

        ranked = shap_rank(X_tr, y_tr, FEATURE_POOL, N_MAX, CONFIG['pool_prefilter'])
        rank_idx = {f: FEATURE_POOL.index(f) for f in ranked}

        for N in CONFIG['n_features_grid']:
            feats = ranked[:N]
            idx = [rank_idx[f] for f in feats]
            n_spike = sum(f in SPIKE_FEATURES for f in feats)
            Xtrk, Xtek = X_tr[:, idx], X_te[:, idx]
            try:
                Xr, yr = BorderlineSMOTE(random_state=SEED, kind='borderline-1').fit_resample(Xtrk, y_tr)
            except Exception:
                Xr, yr = Xtrk, y_tr
            for algo in CONFIG['algos']:
                try:
                    clf = get_clf(algo); clf.fit(Xr, yr)
                    met = metrics(y_te, clf.predict(Xtek))
                except Exception:
                    n_done += 1; continue
                scan_rows.append({'regime': regime, 'N': N, 'algo': algo, 'fold': k + 1,
                                  'n_spike_selected': n_spike, 'n_train': len(y_tr), 'n_test': len(y_te),
                                  'test_start': str(cut_date.date()), 'test_end': str(nxt_date.date()),
                                  'features': ', '.join(feats), **met})
                n_done += 1
        print(f"  fold {k+1} {regime:7s}: {N_MAX} feats rankés, "
              f"grille N×algo faite [{n_done}/{n_total}] ({time.time()-t0:.0f}s)")

df_scan = pd.DataFrame(scan_rows)
print(f"\n[SCAN DONE] {len(df_scan)} évaluations en {(time.time()-t0)/60:.1f}min")


In [ ]:
# ============================================================
# AGRÉGATION WALK-FORWARD + CLASSEMENT DES MEILLEURS MODÈLES
# ============================================================
agg = (df_scan.groupby(['regime', 'N', 'algo'])
       .agg(F1_dir_mean=('F1_dir', 'mean'), F1_dir_std=('F1_dir', 'std'),
            F1_UP_FORT_mean=('F1_UP_FORT', 'mean'), F1_DOWN_FORT_mean=('F1_DOWN_FORT', 'mean'),
            n_spike=('n_spike_selected', 'mean'), n_folds=('fold', 'nunique'))
       .reset_index())
agg = agg[agg['n_folds'] >= 3]  # au moins 3 folds valides pour être crédible
agg = agg.round(4)

# Classement principal : F1_dir moyen (robustesse), puis F1_UP_FORT (l'objectif spike)
best_dir = agg.sort_values('F1_dir_mean', ascending=False).head(15)
best_up = agg.sort_values('F1_UP_FORT_mean', ascending=False).head(15)
# Modèles remplissant les 3 objectifs > 0.50 simultanément en moyenne WF
triple = agg[(agg['F1_dir_mean'] > 0.50) & (agg['F1_UP_FORT_mean'] > 0.50) &
             (agg['F1_DOWN_FORT_mean'] > 0.50)].sort_values('F1_dir_mean', ascending=False)

print("=" * 80)
print(f"SCAN WALK-FORWARD — {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | univers lourd + features spike")
print("=" * 80)
print("\n### TOP 10 par F1_dir moyen (walk-forward) ###")
print(best_dir.head(10).to_string(index=False))
print("\n### TOP 10 par F1_UP_FORT moyen (objectif : fortes hausses) ###")
print(best_up.head(10).to_string(index=False))
print(f"\n### Modèles > 0.50 sur les 3 métriques simultanément (WF) : {len(triple)} ###")
print(triple.head(10).to_string(index=False) if len(triple) else "  aucun")

# Contribution des features spike : les meilleurs modèles en sélectionnent-ils ?
print("\n### Contribution des features spike ###")
for label, tbl in [('Top-10 F1_dir', best_dir.head(10)), ('Top-10 F1_UP_FORT', best_up.head(10))]:
    share = (tbl['n_spike'] > 0).mean()
    print(f"  {label}: {share:.0%} des configs sélectionnent ≥1 feature spike "
          f"(moy. {tbl['n_spike'].mean():.1f} spike/config)")

print("\n### Rappel références rapport (SANS features spike) ###")
for k, v in REPORT_REFS.items():
    print(f"  {k:<38} " + "  ".join(f"{m}={val}" for m, val in v.items()))

# Export
try:
    fname = 'VIX_SPIKE_SCAN_report.xlsx'
    with pd.ExcelWriter(fname, engine='xlsxwriter') as w:
        agg.sort_values('F1_dir_mean', ascending=False).to_excel(w, 'Agg_by_config', index=False)
        best_dir.to_excel(w, 'Top_F1_dir', index=False)
        best_up.to_excel(w, 'Top_F1_UP_FORT', index=False)
        triple.to_excel(w, 'Triple_gt_0.50', index=False)
        df_scan.to_excel(w, 'Detail_per_fold', index=False)
    df_scan.to_csv('vix_spike_scan_detail.csv', index=False)
    print(f"\n[SAVE] {fname} / vix_spike_scan_detail.csv")
except Exception as e:
    print(f"[WARN Export] {e}")
print("\n[NOTE] Aucun modèle figé en production — validation explicite requise (R8).")


## Archivage automatique des résultats vers GitHub (optionnel)
Nécessite le secret Colab `GITHUB_TOKEN` (le même que pour VIX_ML3). Chaque run est archivé dans `results/{version}/{timestamp}/` sur la branche `results/vix-spike-scan` — jamais écrasé.

In [ ]:
# ============================================================
# ARCHIVAGE AUTOMATIQUE DES RÉSULTATS VERS GITHUB (optionnel)
# ============================================================
# Pousse vix_predictions_train/test.parquet, vix_models_metadata.parquet et
# VIX_ML3_stacking_report.xlsx vers une branche dédiée du repo GitHub, pour
# les retrouver sans avoir à les télécharger/uploader manuellement depuis
# Colab. Nécessite un secret Colab nommé GITHUB_TOKEN (icône clé 🔑 dans la
# barre latérale gauche de Colab -> "Ajouter un nouveau secret") contenant un
# Personal Access Token GitHub (fine-grained, scope "Contents: Read and write"
# sur ce seul repo). Si le secret n'est pas configuré, cette cellule ne fait
# rien d'autre qu'un message d'info : aucune conséquence sur le reste du
# notebook.
# [ARCHIVAGE] Chaque run est écrit dans son propre dossier horodaté
# (results/{NOTEBOOK_VERSION}/{timestamp}/) et poussé par un commit normal
# (pas de --force) : les résultats d'un run précédent ne sont jamais
# écrasés, on accumule un historique consultable sur la branche.
import os, subprocess

GITHUB_REPO = "LP-D/claude"          # à adapter si le repo change
RESULTS_BRANCH = "results/vix-spike-scan"
RESULT_FILES = [
    "VIX_SPIKE_SCAN_report.xlsx",
    "vix_spike_scan_detail.csv",
]

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

present = [f for f in RESULT_FILES if os.path.exists(f)]
if not GITHUB_TOKEN:
    print("[SKIP] Secret 'GITHUB_TOKEN' introuvable (Colab: icône clé à gauche > "
          "Ajouter un secret > nom=GITHUB_TOKEN, valeur=ton Personal Access Token GitHub, "
          "accès activé pour ce notebook). Résultats non archivés sur GitHub — "
          "ils restent disponibles en local dans ce runtime Colab.")
elif not present:
    print("[SKIP] Aucun fichier de résultats trouvé à archiver (étapes 1/2 pas encore terminées ?).")
else:
    run_id = f"{NOTEBOOK_VERSION}/{pd.Timestamp.now():%Y%m%d_%H%M%S}"
    workdir = "/content/_vix_ml3_results_push"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(
        ["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git", workdir],
        capture_output=True, text=True)
    if clone.returncode != 0:
        print("[ERREUR] git clone a échoué (token invalide/expiré ou repo inaccessible) :")
        print(clone.stderr[-2000:])
    else:
        # La branche existe peut-être déjà (runs précédents) : on la récupère si oui,
        # sinon on la crée à partir de l'état courant du clone.
        exists = subprocess.run(["git", "-C", workdir, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", workdir, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", workdir, "checkout", "-B", RESULTS_BRANCH], check=True)

        run_dir = f"{workdir}/results/{run_id}"
        os.makedirs(run_dir, exist_ok=True)
        for fn in present:
            subprocess.run(["cp", fn, f"{run_dir}/{fn}"], check=True)
        subprocess.run(["git", "-C", workdir, "config", "user.email", "vix-champion-wf-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", workdir, "config", "user.name", "VIX Champion WF Colab run"], check=True)
        subprocess.run(["git", "-C", workdir, "add", f"results/{run_id}"], check=True)
        commit = subprocess.run(
            ["git", "-C", workdir, "commit", "-m", f"Résultats VIX Champion WF {run_id}"],
            capture_output=True, text=True)
        print(commit.stdout or commit.stderr)
        push = subprocess.run(["git", "-C", workdir, "push", "origin", RESULTS_BRANCH],
                               capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[ARCHIVE OK] Résultats archivés sur la branche '{RESULTS_BRANCH}' de {GITHUB_REPO}, "
                  f"dossier results/{run_id}/ : {len(present)} fichier(s) — {present}")
        else:
            print("[ERREUR] git push a échoué (conflit avec un autre run concurrent ?) :")
            print(push.stderr[-2000:])
